# 03 · ¿Con qué datos se entrenó nuestra primera ACT?

La política `act_cheap_v1` se entrenó en otra máquina y no quedó registro de
qué episodios usó. Eso es grave por una razón concreta: para evaluarla
honestamente necesitamos episodios que **no** haya visto. Si evaluamos sobre
datos de entrenamiento, el número sale lindo y no significa nada.

La pista está en el propio checkpoint: guarda las estadísticas de
normalización del dataset con que se entrenó.

In [ ]:
# torch first: torchvision and torch._dynamo have a circular import that only
# resolves if torch is loaded before lerobot.
import torch  # noqa: F401
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from toolkit.paths import DATASETS_DIR, RAW_RECORDINGS_DIR

plt.rcParams["figure.dpi"] = 110

In [ ]:
from toolkit.checks.normalization import load_policy_stats, load_dataset_stats, compare
from toolkit.dataset.stats import subset_stats
from toolkit.paths import CHECKPOINTS_DIR

POLICY_DIR = CHECKPOINTS_DIR / "act_cheap_v1" / "pretrained_model"
policy_stats = load_policy_stats(POLICY_DIR)
print("cuadros de entrenamiento:", int(policy_stats["index"]["count"][0]))
print("episodios:", int(policy_stats["episode_index"]["max"][0]) + 1)
print("episodio más largo:", int(policy_stats["frame_index"]["max"][0]) + 1, "cuadros")

61 episodios y 22.251 cuadros. Nosotros tenemos 65 grabados. ¿Cuáles son?

Primero, el más largo: 440 cuadros. Busquemos cuál de los nuestros dura eso.

In [ ]:
TASK_DIR = RAW_RECORDINGS_DIR / "pick_mameluco"
lengths = {p.name: len(json.loads((p / "data.json").read_text())["data"])
           for p in sorted(TASK_DIR.iterdir()) if p.is_dir()}
print("total:", sum(lengths.values()), "cuadros en", len(lengths), "episodios")
print("de 440 cuadros:", [k for k, v in lengths.items() if v == 440])

Faltan 22.840 − 22.251 = **589 cuadros** repartidos en 4 episodios. Probamos
todas las combinaciones de 4 (son ~600.000, se hace en segundos).

In [ ]:
import itertools

target = sum(lengths.values()) - int(policy_stats["index"]["count"][0])
candidates = [c for c in itertools.combinations(lengths, 4) if sum(lengths[e] for e in c) == target]
print(target, "cuadros ->", candidates)

Una sola combinación posible, y tiene sentido: tres son los episodios cortitos
del notebook 01. Pero que sumen bien no prueba nada; puede ser casualidad.

**La prueba de verdad**: calcular las estadísticas sin esos 4 episodios y
compararlas con las del modelo. Si es el mismo conjunto, la media y el desvío
de las 26 articulaciones tienen que coincidir hasta el redondeo.

In [ ]:
DATASET_ROOT = DATASETS_DIR / "local" / "pick_mameluco"   # los 65, ver toolkit/README.md
names = sorted(lengths)
exclude = {names.index(e) for e in candidates[0]}

all_65 = load_dataset_stats(DATASET_ROOT)
subset = subset_stats(DATASET_ROOT, exclude)
for label, stats in [("los 65", all_65), ("sin esos 4", subset)]:
    for gap in compare(stats, policy_stats):
        print(f"{label:11s} {gap.feature:18s} corrimiento máx de la media: {gap.max_mean_shift:.2e} desvíos")

In [ ]:
feature = "action"
fig, ax = plt.subplots(figsize=(12, 3))
x = np.arange(26)
ax.bar(x - 0.2, np.abs(all_65[feature]["mean"] - policy_stats[feature]["mean"]) / policy_stats[feature]["std"], 0.4, label="los 65")
ax.bar(x + 0.2, np.abs(subset[feature]["mean"] - policy_stats[feature]["mean"]) / policy_stats[feature]["std"], 0.4, label="sin esos 4")
ax.set_yscale("log"); ax.set_ylabel("|Δ media| / desvío"); ax.set_xlabel("articulación"); ax.legend()
plt.tight_layout()

Con los 65 la diferencia es de centésimas de desvío; sin esos 4 baja a 1e-8,
que es el redondeo de float32. **Es el mismo conjunto.**

¿Y la cámara? Sus estadísticas en el modelo son las de ImageNet, no las del
dataset: LeRobot hace eso por defecto (`use_imagenet_stats`) porque el
backbone ResNet18 viene preentrenado con esa normalización.

In [ ]:
img = policy_stats["observation.images.cam_right_high"]
print("media:", img["mean"].ravel(), " desvío:", img["std"].ravel())

## El split

El config de entrenamiento lista los 54 episodios usados; los otros 7 (índices
0, 10, …, 60) quedaron afuera. Eso queda escrito en `learning/act/split.json`
y es lo que va a usar la evaluación offline.

In [ ]:
from toolkit.paths import REPO_ROOT
split = json.loads((REPO_ROOT / "learning" / "act" / "split.json").read_text())
print("train:", len(split["train_episode_indices"]), " held-out:", split["heldout_raw_episodes"])